# Shrink the Model: Export to ONNX + Quantize to INT8 (v2 — No `optimum`)

**Why this notebook exists:**
`bert-base-multilingual-cased` has a ~120,000-token vocabulary (covering 100+ languages).
The embedding table alone is ~370MB, and the full model is **~711MB** in full precision.
Render's free tier has a hard limit of **512MB RAM**, so the full model causes an immediate **OOM (Out Of Memory) crash**, resulting in `502 Bad Gateway`.

**Why v2 (No `optimum`):**
The previous version used `optimum[onnxruntime]`, which clashed with Google Colab's pre-installed `diffusers` and `gradio` libraries by downgrading `huggingface-hub`.
This version uses only:
- PyTorch's native `torch.onnx.export` (already in Colab, no install needed)
- `onnx` & `onnxruntime` for INT8 dynamic quantization
- Standard `huggingface_hub` for pushing the quantized model

Run each cell in order. CPU runtime is fine (GPU not needed).


## Step 1 — Install ONNX tools (safe, no dependency conflict)


In [ ]:
!pip install -q onnx onnxruntime


## Step 2 — Load your trained model from Hugging Face Hub


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "lohithg8408/content-moderation"
SEQ_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.eval()

print("Loaded model:", MODEL_ID)
print("Label mapping:", model.config.id2label)


## Step 3 — Export to ONNX using PyTorch's built-in exporter
We include dynamic axes for batch_size and sequence_length so the model can handle any batch size or text length.


In [ ]:
dummy = tokenizer(
    "dummy text for export",
    return_tensors="pt",
    padding="max_length",
    max_length=SEQ_LEN,
    truncation=True,
)

torch.onnx.export(
    model,
    (dummy["input_ids"], dummy["attention_mask"], dummy["token_type_ids"]),
    "model_fp32.onnx",
    input_names=["input_ids", "attention_mask", "token_type_ids"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "token_type_ids": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"},
    },
    opset_version=14,
)

print("Exported FP32 ONNX model. File size:")
!du -sh model_fp32.onnx


## Step 4 — Quantize to INT8 (targeting both MatMul and Gather)
`Gather` is the embedding lookup operation. Quantizing both `MatMul` and `Gather` shrinks both the transformer layers AND the huge 370MB multilingual embedding table.


In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="model_fp32.onnx",
    model_output="model_quantized.onnx",
    op_types_to_quantize=["MatMul", "Gather"],
    weight_type=QuantType.QInt8,
)

print("Quantized INT8 ONNX model. File size:")
!du -sh model_quantized.onnx


## Step 5 — Verify the quantized size
The quantized model should now be ~150–200MB (down from ~711MB), fitting comfortably within Render's 512MB RAM budget!


In [ ]:
import os
fp32_size = os.path.getsize("model_fp32.onnx") / (1024 * 1024)
int8_size = os.path.getsize("model_quantized.onnx") / (1024 * 1024)
print(f"FP32 Model: {fp32_size:.1f} MB")
print(f"INT8 Model: {int8_size:.1f} MB")
print(f"Reduction: {((fp32_size - int8_size) / fp32_size) * 100:.1f}%")


## Step 6 — Sanity check: verify model predictions


In [ ]:
import numpy as np
import onnxruntime as ort

session = ort.InferenceSession("model_quantized.onnx", providers=["CPUExecutionProvider"])
id2label = model.config.id2label

def classify(text):
    inputs = tokenizer(
        text, return_tensors="np", padding=True, truncation=True, max_length=SEQ_LEN
    )
    onnx_inputs = {
        "input_ids": inputs["input_ids"].astype(np.int64),
        "attention_mask": inputs["attention_mask"].astype(np.int64),
        "token_type_ids": inputs["token_type_ids"].astype(np.int64),
    }
    logits = session.run(None, onnx_inputs)[0][0]
    probs = np.exp(logits - np.max(logits))
    probs = probs / probs.sum()
    idx = int(np.argmax(probs))
    return id2label[idx], float(probs[idx])

for sample in ["you are such an idiot", "have a great day everyone", "get out of here you trash"]:
    label, conf = classify(sample)
    print(f"{sample!r} -> {label} ({conf:.2%})")


## Step 7 — Package model, tokenizer, and config


In [ ]:
import os
import shutil

OUT_DIR = "onnx_model_quantized"
os.makedirs(OUT_DIR, exist_ok=True)

shutil.copy("model_quantized.onnx", f"{OUT_DIR}/model.onnx")
tokenizer.save_pretrained(OUT_DIR)
model.config.save_pretrained(OUT_DIR)

print("Export directory contents:")
!ls -lh {OUT_DIR}


## Step 8 — Login to Hugging Face
Generate a token with **WRITE** access at https://huggingface.co/settings/tokens, paste it below, and click Login.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## Step 9 — Push the quantized model to Hugging Face Hub


In [ ]:
from huggingface_hub import HfApi, create_repo

NEW_REPO = "lohithg8408/content-moderation-onnx-int8"

create_repo(NEW_REPO, exist_ok=True)
api = HfApi()
api.upload_folder(folder_path=OUT_DIR, repo_id=NEW_REPO)
print(f"\nSuccessfully pushed to: https://huggingface.co/{NEW_REPO}")
